In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'plotly', '-q'])

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df1 = pd.read_csv('Unemployment in India.csv')
df2 = pd.read_csv('Unemployment_Rate_upto_11_2020.csv')

df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()
df1['Date'] = pd.to_datetime(df1['Date'].str.strip(), dayfirst=True)
df2['Date'] = pd.to_datetime(df2['Date'].str.strip(), dayfirst=True)

df_all = pd.concat([df1, df2], ignore_index=True)
df_all['Date'] = pd.to_datetime(df_all['Date'], dayfirst=True, errors='coerce')
df_all = df_all.dropna(subset=['Date'])
df_all['Month'] = df_all['Date'].dt.to_period('M')
df_all['COVID'] = df_all['Date'].apply(
    lambda x: 'During COVID' if x >= pd.Timestamp('2020-03-01') else 'Pre COVID'
)

col_ur  = 'Estimated Unemployment Rate (%)'
col_lpr = 'Estimated Labour Participation Rate (%)'

print(f"✅ Data Loaded! Records: {len(df_all)}, States: {df_all['Region'].nunique()}")

✅ Data Loaded! Records: 1007, States: 28


In [2]:
pre  = df_all[df_all['COVID']=='Pre COVID'][col_ur].mean()
dur  = df_all[df_all['COVID']=='During COVID'][col_ur].mean()
peak = df_all[col_ur].max()
peak_state = df_all.loc[df_all[col_ur].idxmax(), 'Region']

print("="*50)
print(f"  Avg Unemployment Pre-COVID    : {pre:.2f}%")
print(f"  Avg Unemployment During COVID : {dur:.2f}%")
print(f"  Increase due to COVID         : +{dur-pre:.2f}%")
print(f"  Peak Unemployment             : {peak:.2f}% ({peak_state})")
print("="*50)

  Avg Unemployment Pre-COVID    : 9.48%
  Avg Unemployment During COVID : 15.31%
  Increase due to COVID         : +5.82%
  Peak Unemployment             : 76.74% (Puducherry)


In [4]:
import plotly.graph_objects as go
import pandas as pd

BG=  '#0d1117'; CARD='#161b22'; BLUE='#58a6ff'
RED= '#f78166'; GREEN='#3fb950'; TEXT='#e6edf3'

monthly = df_all.groupby('Month')[col_ur].mean().reset_index()
monthly['Date'] = monthly['Month'].dt.to_timestamp()
monthly = monthly.sort_values('Date')

covid_date = monthly[monthly['Date'] >= '2020-03-01']['Date'].min()

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=monthly['Date'], y=monthly[col_ur],
    fill='tozeroy', fillcolor='rgba(88,166,255,0.15)',
    line=dict(color=BLUE, width=2.5), mode='lines+markers', marker=dict(size=5)))
fig1.add_vline(x=covid_date.timestamp()*1000, line_dash='dash', line_color=RED,
               annotation_text='COVID-19 Onset', annotation_font_color=RED)
fig1.update_layout(title='📈 Monthly Unemployment Rate Trend',
    plot_bgcolor=CARD, paper_bgcolor=BG, font_color=TEXT, height=420,
    xaxis=dict(gridcolor='#21262d'), yaxis=dict(gridcolor='#21262d'))
fig1.show()

In [5]:
fig2 = go.Figure()
fig2.add_trace(go.Box(y=df_all[df_all['COVID']=='Pre COVID'][col_ur],
    name='Pre COVID', marker_color=GREEN, line_color=GREEN))
fig2.add_trace(go.Box(y=df_all[df_all['COVID']=='During COVID'][col_ur],
    name='During COVID', marker_color=RED, line_color=RED))
fig2.update_layout(title='📦 Pre vs During COVID Comparison',
    plot_bgcolor=CARD, paper_bgcolor=BG, font_color=TEXT, height=420,
    yaxis=dict(gridcolor='#21262d'))
fig2.show()

In [6]:
state_avg = df_all.groupby('Region')[col_ur].mean().sort_values(ascending=False).head(10)
bar_colors = [RED if i < 3 else BLUE for i in range(len(state_avg))]

fig3 = go.Figure(go.Bar(
    x=state_avg.values[::-1], y=state_avg.index[::-1], orientation='h',
    marker_color=bar_colors[::-1],
    text=[f'{v:.1f}%' for v in state_avg.values[::-1]], textposition='outside'))
fig3.update_layout(title='🏆 Top 10 States by Unemployment Rate',
    plot_bgcolor=CARD, paper_bgcolor=BG, font_color=TEXT,
    height=450, margin=dict(l=160, r=80))
fig3.show()

In [7]:
if 'Area' in df1.columns:
    area_data = df1.groupby('Area')[col_ur].mean()
    fig4 = go.Figure(go.Pie(labels=area_data.index, values=area_data.values,
        marker=dict(colors=[BLUE, '#ffa657'], line=dict(color=BG, width=2)), hole=0.4))
    fig4.update_layout(title='🥧 Rural vs Urban Unemployment',
        paper_bgcolor=BG, font_color=TEXT, height=420)
    fig4.show()

In [9]:
lpr_m = df_all.groupby('Month')[col_lpr].mean().reset_index()
lpr_m['Date'] = lpr_m['Month'].dt.to_timestamp()
lpr_m = lpr_m.sort_values('Date')

covid_date2 = lpr_m[lpr_m['Date'] >= '2020-03-01']['Date'].min()

fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=lpr_m['Date'], y=lpr_m[col_lpr],
    fill='tozeroy', fillcolor='rgba(63,185,80,0.15)',
    line=dict(color=GREEN, width=2.5), mode='lines+markers', marker=dict(size=5)))
fig5.add_vline(x=covid_date2.timestamp()*1000, line_dash='dash', line_color=RED,
               annotation_text='COVID-19 Onset', annotation_font_color=RED)
fig5.update_layout(title='👷 Labour Participation Rate Trend',
    plot_bgcolor=CARD, paper_bgcolor=BG, font_color=TEXT, height=420,
    xaxis=dict(gridcolor='#21262d'), yaxis=dict(gridcolor='#21262d'))
fig5.show()

In [10]:
# COVID Impact
impact = df_all.groupby(['Region','COVID'])[col_ur].mean().unstack()
impact['change'] = impact['During COVID'] - impact['Pre COVID']
top_impact = impact['change'].dropna().sort_values(ascending=False).head(8)
impact_colors = [RED if v > 0 else GREEN for v in top_impact.values]

fig6 = go.Figure(go.Bar(
    x=top_impact.values[::-1], y=top_impact.index[::-1], orientation='h',
    marker_color=impact_colors[::-1],
    text=[f'+{v:.1f}%' if v>0 else f'{v:.1f}%' for v in top_impact.values[::-1]],
    textposition='outside'))
fig6.update_layout(title='🔥 COVID Impact by State',
    plot_bgcolor=CARD, paper_bgcolor=BG, font_color=TEXT,
    height=420, margin=dict(l=160, r=80))
fig6.show()

# Heatmap
heat_data = df_all.groupby(['Region','Month'])[col_ur].mean().unstack()
heat_data = heat_data.loc[heat_data.mean(axis=1).sort_values(ascending=False).head(12).index]
heat_data = heat_data.fillna(0)
heat_data.columns = [str(c) for c in heat_data.columns]

fig7 = go.Figure(go.Heatmap(
    z=heat_data.values.tolist(), x=heat_data.columns.tolist(),
    y=heat_data.index.tolist(), colorscale='RdYlGn_r', zmin=0))
fig7.update_layout(title='🗺️ State-wise Unemployment Heatmap',
    plot_bgcolor=CARD, paper_bgcolor=BG, font_color=TEXT,
    height=500, margin=dict(l=130, r=20, b=120),
    xaxis=dict(tickangle=-45, tickfont=dict(size=9)))
fig7.show()

print("\n✅ All Done!")


✅ All Done!
